In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from collections import defaultdict

#custom functions
from heuristic_functions import *

In [2]:
%run initialize_heuristic_data.py

In [ ]:
%who


In [4]:
# Selecting only the rows for 'Measles', 'Mumps', and 'Rubella' in both datasets
demand_80_MCV = demand_80_final
interim_demand_DF = demand_80_MCV.copy()
tender_schedules_MCV = tender_schedules
# i_start_MCV = i_start[i_start['Vaccine'].isin(['M', 'MR', 'MMR'])]
missed_doses_MCV = missed_doses
ratio_DF_MCV = pd.DataFrame()
inventory_DF_MCV = inventory_DF

#logic to setup least covered antigens:
# Flatten the list of all antigens from all vaccines
all_antigens = [antigen for antigens in A_v.values() for antigen in antigens]
# Count the occurrences of each antigen
antigen_counts = Counter(all_antigens)
# least_covered_antigens = sorted(antigen_counts.keys(), key=lambda x: antigen_counts[x], reverse=False)



In [ ]:
# inventory_DF_MCV.loc[inventory_DF_MCV['Vaccine']=='HPV', 'Amount'] = 0
demand_80_MCV

In [6]:
def fulfill_demand2(vaccine_price_list, antigen, interim_demand, manufacturer_capacities_df, inventory_DF, year, total_price, A_v):
    if year not in manufacturer_capacities_df.columns:
        print(f"Year {year} is not valid. Exiting function.")
        return {
            'Inventory_Used': [],
            'Demand_Filled': 0,
            'Demand_unfilled': [],
            'Total_Price': total_price,
            'Message': "Invalid year",
            'Years_Tendered': 0  # Ensure we return 0 years tendered if year is invalid
        }

    vaccine_price_list.sort(key=lambda x: x['Price'])
    total_demand_filled = 0
    demand_unfilled = []
    inventory_used = []
    years_tendered = 0  # Track how many years the demand was fulfilled

    for entry in vaccine_price_list:
        manufacturer, vaccine, price = entry['Manufacturer'], entry['Vaccine'], entry['Price']

        if antigen not in interim_demand['antigen'].values:
            raise ValueError(f"Antigen '{antigen}' not found.")

        antigen_demand = interim_demand.loc[interim_demand['antigen'] == antigen, year].iloc[0]
        if antigen_demand == 0:
            years_tendered += 1
            continue

        manufacturer_row = manufacturer_capacities_df.loc[manufacturer_capacities_df['Manufacturer'] == manufacturer, year]
        if manufacturer_row.empty:
            raise ValueError(f"Manufacturer '{manufacturer}' not found.")
        
        capacity = manufacturer_row.iloc[0]
        if capacity < antigen_demand:
            print(f"Insufficient capacity for {antigen} in year {year}. Exiting tender.")
            return {
                'Inventory_Used': inventory_used,
                'Demand_Filled': total_demand_filled,
                'Demand_unfilled': demand_unfilled,
                'Total_Price': total_price,
                'Message': "Insufficient capacity",
                'Years_Tendered': years_tendered  # Return the number of years tendered before exit
            }

        amount_to_fill = min(antigen_demand, capacity)

        interim_demand.loc[interim_demand['antigen'] == antigen, year] -= amount_to_fill
        inventory_DF.loc[inventory_DF['Vaccine'] == vaccine, 'Amount'] += amount_to_fill
        manufacturer_capacities_df.loc[manufacturer_capacities_df['Manufacturer'] == manufacturer, year] -= amount_to_fill

        for ant in A_v.get(vaccine, []):
            interim_demand.loc[interim_demand['antigen'] == ant, year] -= amount_to_fill

        inventory_used.append({manufacturer: amount_to_fill})
        total_demand_filled += amount_to_fill
        total_price += amount_to_fill * price

        years_tendered += 1  # Increment each year the demand is fulfilled

    if interim_demand.loc[interim_demand['antigen'] == antigen, year].iloc[0] > 0:
        demand_unfilled.append(antigen)

    message = ["Demand fully fulfilled" if antigen not in demand_unfilled else "Demand not fully fulfilled"]

    return {
        'Inventory_Used': inventory_used,
        'Demand_Filled': total_demand_filled,
        'Demand_unfilled': demand_unfilled,
        'Total_Price': total_price,
        'Message': message,
        'Years_Tendered': years_tendered  # Return the number of years the tender was fulfilled
    }



In [ ]:
for year in range(1, 11):  # Iterate through each year - short range for testing  range(1,len(demand_80_MCV.columns)-1)
    print("********************HAPPY NEW YEAR****************************")
    print("******************Reticulating splines...**************************")
    print(f"Year: {year}")

    least_covered_antigens = sorted(antigen_counts.keys(), key=lambda x: antigen_counts[x], reverse=False)
    remaining_inventory = {}
    uncovered_demand = {}
    # print(f"least covered antigens: {least_covered_antigens}")
    while least_covered_antigens:  # Iterate through each antigen, find what vaccines cover each antigen, least to greatest, update supply and demand
        print('###############################################################')
        antigen = least_covered_antigens.pop(0)
        print(f"serving antigen {antigen}")
        # print(f"Initial Demand: {demand_80_MCV.loc[demand_80_MCV['antigen'] == antigen, year].iloc[0]} for {antigen}")
        # print(f"Initial Inventory of vaccine covering {antigen}: {inventory_DF_MCV}")
        # print(f"A_v items are: {A_v.items()}")
        for vaccine, antigens in A_v.items():  # Iterate through A_v to check which vaccines cover the antigen
            # print(f"antigens list is: {antigens}")
            if antigen in antigens and demand_80_MCV.loc[demand_80_MCV['antigen'] == antigen, year].iloc[0] > 0:
                print(f"{antigen} FOUND in {vaccine}")
                vaccine_inventory_value = inventory_DF_MCV.loc[inventory_DF_MCV['Vaccine'] == vaccine, 'Amount'].iloc[0]
                # print("iiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii")
                # print(f"Inventory for {vaccine} for year {year}: ", vaccine_inventory_value)

                antigen_demand_value = demand_80_MCV.loc[demand_80_MCV['antigen'] == antigen, year].iloc[0]
                # print(f"Demand value for {antigen} for year {year}: ", antigen_demand_value)

                difference = vaccine_inventory_value - antigen_demand_value
                # print(f"The difference between supply and demand is : {difference}")
                if difference >= 0: #
                    remaining_inventory[vaccine] = difference
                    decrement = antigen_demand_value
                else: 
                    remaining_inventory[vaccine] = 0
                    decrement = vaccine_inventory_value
                    uncovered_demand[antigen] = abs(difference)
                    # print("------------------------------------------")
                    # print(f"Vaccine:3 {vaccine}, antigen: {antigen}")
                    # print(f"uncovered demand for {antigen}: {uncovered_demand[antigen]}")
                    #transfer uncovered demand to next year

                print("^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^")
                inventory_DF_MCV.loc[inventory_DF_MCV.iloc[:, 0] == vaccine, 'Amount'] = remaining_inventory[vaccine]
                # print(f"Adjusted Inventory of {vaccine} covering {antigen}: {inventory_DF_MCV}")

                print("Decrementing antigen demands")
                for ant in antigens:
                    print(f"antigen to decrement: {ant}")
                    current_demand = demand_80_MCV.loc[demand_80_MCV["antigen"] == ant, year].iloc[0]
                    if current_demand > 0:
                        # print(f"pre-decrement {ant} demand: {demand_80_MCV.loc[demand_80_MCV['antigen'] == ant, year].iloc[0]}")
                        # print(f"From {ant} demand, reducing demand for year {year} for antigen {ant} by {decrement}")
                        # Update the demand for the antigen by subtracting the decrement in a single step
                        demand_80_MCV[year] = demand_80_MCV[year].astype(float)
                        demand_80_MCV.loc[demand_80_MCV["antigen"] == ant, year] = current_demand - decrement
                        # print(f"post decrement {ant} demand: {demand_80_MCV.loc[demand_80_MCV['antigen'] == ant, year].iloc[0]}")
            elif antigen in antigens and demand_80_MCV.loc[demand_80_MCV['antigen'] == antigen, year].iloc[0] <= 0:
                print(f"Demand for {antigen} is {demand_80_MCV.loc[demand_80_MCV['antigen'] == antigen, year].iloc[0]}!")
            else:
                print(f"{antigen} not in {vaccine}")


        print(f"{len(antigen_counts)} antigens entered, only {len(least_covered_antigens)} remain!")

    #update any uncovered demand, to next year. add uncovered demand to dosses_missed dict
    if 'uncovered_demand' in locals(): # Check if the variable exists
        while uncovered_demand:
            top = uncovered_demand.popitem()
            top_antigen = top[0]
            doses_missed = top[1]
            # print(f"{doses_missed} doses missed for {top_antigen}")
            demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == top_antigen, demand_80_MCV.columns[year+1]] += float(doses_missed)
            missed_doses_MCV.loc[missed_doses_MCV.iloc[:, 0] == top_antigen, missed_doses_MCV.columns[1]] += float(doses_missed)
            ###################################################################################################################
            #Need to add logic here to update interim_demand_DF[year +1]  with any uncovered demand
            interim_demand_DF.loc[interim_demand_DF.iloc[:,0]==top_antigen, demand_80_MCV.columns[year+1]] += float(doses_missed)
    else:
        print("no uncovered demand this year")
    
    #check ratio for supply/demand.
    #check at end of year for math reasons. if ratio is less than 1, schedule tender, perform search for vaccines, add inventory
    # print(f"Checking ratio of supply to demand for antigens for year {year + 1}!")
    # print()
    #pulls the current ratio of supply and demand. returns ratio_DF and antigen coverage DF
    ratio_DF_MCV, coverage_df = calculate_coverage_and_ratios(inventory_DF_MCV, demand_80_MCV, V_a, year+1)

    ratio_DF_MCV = ratio_DF_MCV.sort_values(by='antigen', key=lambda x: x.map(antigen_counts), ascending=True)

    for index, row in ratio_DF_MCV.iterrows():
        if row['Ratio'] < 1.0: #create three year tender
            #update:
            # total_price += tender_cost
            # print(f"Ratio: {round(row.loc['Ratio'],2)}")
            print(f"Generating Tender for {row.loc['antigen']}")
            #append F schedule for curreny year +1 to current year +1 + tender_length

            for time_period in range(1,max_tender_length+1):
                print("*^*^*^*^*^ TENDERING *^*^*^*^*^*^*^*^*^")
                print(f"TENDER PLANNING FOR YEAR: {year + 1}")
                #inventory search
                price_list = get_manufacturer_vaccine_price(row.loc['antigen'], price_data, V_a, P_v, year + 1)
                print(f"Lowest Price Information: {price_list}")
                # print(f"Pre fulfillment inventory: {inventory_DF_MCV}")
                # print("")
                # print(f"pre capacity_data: {capacity_data.loc[capacity_data['Manufacturer']==maunfacturer, year]}")
                result = fulfill_demand2(price_list, row.loc['antigen'], interim_demand_DF, capacity_data, inventory_DF_MCV, year+time_period, total_price, A_v)
                # total_price += result['Total_Price']           
                print(result)
            new_row = {'Antigen': row.loc['antigen'], 'Starting': year+1, 'Ending': year + result['Years_Tendered'] }
            tender_schedules_MCV = pd.concat([tender_schedules_MCV, pd.DataFrame([new_row])], ignore_index=True)


        else: #ratio greater than 1
            print(f"Ratio: {round(row.loc['Ratio'],2)}")
            print(f"Supply >= demand for {row.loc['antigen']}")


In [ ]:
price_data